In [1]:
# 🏀 March Madness 2025 - Enhanced Model with Fixes
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames"]
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])

    return team_stats

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    # ✅ Merge seeds
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

    # ✅ Merge team stats
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # ✅ Convert non-numeric values to NaN and fill them
    results = results.apply(pd.to_numeric, errors="coerce")
    results.fillna(0, inplace=True)

    # ✅ Feature Engineering
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]
    results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]

    # ✅ Create dataset with both win/loss scenarios
    win_features = results[["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]].copy()
    win_features["Win"] = 1  

    loss_features = results[["LTeamID", "WTeamID", "LSeed", "WSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]].copy()
    loss_features.columns = ["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Train & Evaluate Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

best_model = None
best_score = float("inf")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    score = log_loss(y_valid, y_pred)
    print(f"{name} Log Loss: {score}")
    if score < best_score:
        best_score = score
        best_model = model

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores)}")

# 🏀 Generate Tournament Predictions
submission_df = pd.read_csv(os.path.join(data_path, "SampleSubmissionStage2.csv"))

# ✅ Extract Season, WTeamID, and LTeamID
submission_df[["Season", "WTeamID", "LTeamID"]] = submission_df["ID"].str.split("_", expand=True).astype(int)

# ✅ Merge seed information
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])

# ✅ Merge team statistics
submission_df = submission_df.merge(mteam_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
submission_df = submission_df.merge(mteam_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

# ✅ Compute matchup features
submission_df["SeedDiff"] = submission_df["WSeed"] - submission_df["LSeed"]
submission_df["WinRateDiff"] = submission_df["WinRate_W"] - submission_df["WinRate_L"]
submission_df["ScoreMarginDiff"] = submission_df["ScoreMargin_W"] - submission_df["ScoreMargin_L"]

# ✅ Prepare features for model prediction
submission_features = submission_df[["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]]

# ✅ Ensure consistency in feature names
X_columns = list(X.columns)  # Get the exact feature names used during training
submission_features = submission_features[X_columns]  # Select only matching columns

# ✅ Scale features
submission_features_scaled = scaler.transform(submission_features)

# ✅ Generate predictions
submission_df["Pred"] = best_model.predict_proba(submission_features_scaled)[:, 1]

# ✅ Save submission file
submission_df[["ID", "Pred"]].to_csv("submission.csv", index=False)

print("✅ Submission file generated successfully!")


Logistic Regression Log Loss: 0.5144581574610574
Random Forest Log Loss: 0.10693468218365951
Gradient Boosting Log Loss: 0.044028825154777844
XGBoost Log Loss: 0.0416901580803092
Cross-Validation Log Loss: 0.04734392366501371
✅ Submission file generated successfully!


 * there were some underlying issues that could lead to data leakage, incorrect feature processing, and missing values.

In [3]:
print("Training Feature Columns:", X.columns.tolist())
print("Submission Feature Columns:", submission_features.columns.tolist())

print("Mean of X_train before scaling:", X_train.mean())
print("Mean of X_train_scaled:", X_train_scaled.mean())

print("Best Model Parameters:", best_model.get_params())

print("NaN values in submission features:", submission_features.isna().sum())


Training Feature Columns: ['WTeamID', 'LTeamID', 'WSeed', 'LSeed', 'SeedDiff', 'WinRateDiff', 'ScoreMarginDiff']
Submission Feature Columns: ['WTeamID', 'LTeamID', 'WSeed', 'LSeed', 'SeedDiff', 'WinRateDiff', 'ScoreMarginDiff']
Mean of X_train before scaling: WTeamID            2076.106479
LTeamID            2076.052639
WSeed                 6.686563
LSeed                 6.707858
SeedDiff             -4.014997
WinRateDiff           0.008062
ScoreMarginDiff      96.543341
dtype: float64
Mean of X_train_scaled: 1.8419674142379836e-17
Best Model Parameters: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_thresho

In [7]:
# 🏀 March Madness 2025 - Enhanced Model with Final Optimizations
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

# 📂 Define the data path
data_path = "data/"

# 🏀 Load Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames"]
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])

    return team_stats

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# 🏀 Prepare Training Data
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # ✅ Ensure features exist
    results["WSeed"] = results["WSeed"].fillna(results["WSeed"].median())
    results["LSeed"] = results["LSeed"].fillna(results["LSeed"].median())
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]

    if "WinRate_W" in results.columns and "WinRate_L" in results.columns:
        results["WinRateDiff"] = results["WinRate_W"] - results["WinRate_L"]
    else:
        results["WinRateDiff"] = 0 

    if "ScoreMargin_W" in results.columns and "ScoreMargin_L" in results.columns:
        results["ScoreMarginDiff"] = results["ScoreMargin_W"] - results["ScoreMargin_L"]
    else:
        results["ScoreMarginDiff"] = 0 

    feature_columns = ["WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]
    
    win_features = results[feature_columns].copy()
    win_features["Win"] = 1  

    loss_features = results[feature_columns].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Train & Evaluate Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_split=8),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, min_samples_leaf=3),
    "XGBoost": XGBClassifier(n_estimators=250, learning_rate=0.05, max_depth=6, reg_lambda=1.0, gamma=0.5, subsample=0.8)
}

best_model = None
best_score = float("inf")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    
    log_loss_score = log_loss(y_valid, y_pred)
    brier_score = brier_score_loss(y_valid, y_pred)
    
    print(f"{name} Log Loss: {log_loss_score:.6f}")
    print(f"{name} Brier Score: {brier_score:.6f}\n{'-'*50}")
    
    if log_loss_score < best_score:
        best_score = log_loss_score
        best_model = model

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores):.6f}")

print("✅ Model Training Completed Successfully!")


Logistic Regression Log Loss: 0.693293
Logistic Regression Brier Score: 0.250073
--------------------------------------------------
Random Forest Log Loss: 0.730267
Random Forest Brier Score: 0.268434
--------------------------------------------------
Gradient Boosting Log Loss: 0.788113
Gradient Boosting Brier Score: 0.295395
--------------------------------------------------
XGBoost Log Loss: 0.835928
XGBoost Brier Score: 0.319182
--------------------------------------------------
Cross-Validation Log Loss: 0.693798
✅ Model Training Completed Successfully!
